In [ ]:
import pandas as pd
from pathlib import Path
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Locate jobs_clean.csv
jobs_path = Path("data/processed/jobs_clean.csv")
if not jobs_path.exists():
    jobs_path = Path("../data/processed/jobs_clean.csv")

jobs = pd.read_csv(jobs_path)
print(f"Loaded {len(jobs)} job records.")

# Vectorize the combined text (title, skills, description)
tfidf = TfidfVectorizer(
    max_features=5000,
    stop_words="english",
    ngram_range=(1, 2)
)
job_matrix = tfidf.fit_transform(jobs["text"].fillna(""))
print("Job TF-IDF matrix shape:", job_matrix.shape)

Loaded 20000 job records.
Job TF-IDF matrix shape: (20000, 5000)


In [ ]:
def recommend_jobs(resume_text: str, top_n: int = 5) -> pd.DataFrame:
    # Transform resume into the same 5000-feature space
    resume_vec = tfidf.transform([resume_text.lower()])
    
    # Compute cosine similarity 
    sim_scores = cosine_similarity(resume_vec, job_matrix).flatten()
    
    # Indices of top-N scores
    top_indices = sim_scores.argsort()[-top_n:][::-1]
    
    # Results table
    recommendations = jobs.iloc[top_indices][["title", "company", "location", "skills"]].copy()
    recommendations["similarity_score"] = (sim_scores[top_indices] * 100).round(2)
    
    return recommendations

In [3]:
# Profile A: Data Science
ds_profile = """
Proficient in Python, SQL, machine learning, Pandas, NumPy, Scikit-Learn,
predictive modeling, data visualization, and exploratory data analysis.
"""
print("Top Matches for Data Science Profile:")
display(recommend_jobs(ds_profile, top_n=5))

# Profile B: Java Developer
java_profile = """
Backend developer experienced with Java, Spring Boot, microservices, REST APIs,
Hibernate, MySQL, and Docker containerization.
"""
print("\nTop Matches for Java Developer Profile:")
display(recommend_jobs(java_profile, top_n=5))

Top Matches for Data Science Profile:


,title,company,location,skills,similarity_score
1477,Lead Machine Learning Engineer,NaN,United States,NaN,53.99
3683,Machine Learning Engineer - Remote,NaN,"Austin, TX",NaN,48.16
19788,Machine Learning Engineer,NaN,"California, United States",NaN,45.37
335,Senior Machine Learning Engineer,NaN,"Los Angeles, CA",NaN,44.49
12089,Sr GenAI / Machine Learning Engineer,NaN,United States,NaN,44.13



Top Matches for Java Developer Profile:


,title,company,location,skills,similarity_score
16136,Senior Java Backend Developer,NaN,"Santa Clara, CA",NaN,49.91
19324,Lead Java Backend Developer with AWS Certified,NaN,"Plano, TX",NaN,43.23
2672,JAVA FULLSTACK DEVELOPER,NaN,"Charlotte, NC",NaN,43.19
1357,Senior Java Developer,NaN,United States,NaN,39.69
19628,Java Fullstack Developer,NaN,"Plano, TX",NaN,39.33


In [4]:
import joblib

models_dir = Path("models")
if not models_dir.exists():
    models_dir = Path("../models")
models_dir.mkdir(parents=True, exist_ok=True)

joblib.dump(tfidf, models_dir / "job_tfidf_vectorizer.pkl")
print("Saved recommender vectorizer to models/job_tfidf_vectorizer.pkl")

Saved recommender vectorizer to models/job_tfidf_vectorizer.pkl
